In [1]:
import math
import time
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import urllib.request


In [2]:
@dataclass
class Config:
    # data
    book_path: str = "book.txt"
    block_size: int = 96
    train_split: float = 0.9

    # model
    vocab_size: int = 0
    d_model: int = 96
    n_heads: int = 4
    n_layers: int = 2
    d_ff: int = 4 * 96
    dropout: float = 0.1

    # training
    batch_size: int = 64
    lr: float = 3e-4
    weight_decay: float = 1e-2
    max_epochs: int = 1
    eval_every: int = 200
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # quantization-aware training
    use_fake_quant: bool = True
    fake_quant_bits: int = 8

    # generation
    max_new_tokens: int = 120
    temperature: float = 0.9
    top_k: int = 40

    # reproducibility
    seed: int = 42

In [ ]:
def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
def count_parameters(model: nn.Module):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
def estimate_model_size_mb(model: nn.Module):
    total_bytes = 0
    for p in model.state_dict().values():
        total_bytes += p.numel() * p.element_size()
    return total_bytes / (1024 ** 2)

In [3]:
class CharTokenizer:
    def __init__(self, text: str):
        chars = sorted(list(set(text)))
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}
        self.vocab_size = len(chars)

    def encode(self, s: str):
        return [self.stoi[c] for c in s]

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)
    
class BookDataset(Dataset):
    def __init__(self, token_ids, block_size):
        self.data = torch.tensor(token_ids, dtype=torch.long)
        self.block_size = block_size

    def __len__(self):
        return max(0, len(self.data) - self.block_size)

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.block_size]
        y = self.data[idx + 1: idx + self.block_size + 1]
        return x, y

In [4]:

def fake_quantize_tensor(x: torch.Tensor, num_bits: int = 8, eps: float = 1e-8):
    """
    Symmetric fake quantization:
      scale = max(|x|) / qmax
      q = clamp(round(x / scale), qmin, qmax)
      x_hat = q * scale
    where qmax = 2^(num_bits - 1) - 1 and qmin = -2^(num_bits - 1)
    """
    if not x.is_floating_point():
        return x

    qmax = 2 ** (num_bits - 1) - 1
    qmin = -2 ** (num_bits - 1)

    max_abs = x.detach().abs().max()
    scale = max_abs / max(qmax, 1)
    scale = torch.clamp(scale, min=eps)

    q = torch.clamp(torch.round(x / scale), qmin, qmax)
    x_hat = q * scale
    return x_hat
class FakeQuantLinear(nn.Module):
    """
    Drop-in linear layer with optional fake quantization of
    input activations and weights.
    """
    def __init__(self, in_features, out_features, bias=True, num_bits=8, enable_fake_quant=True):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self.num_bits = num_bits
        self.enable_fake_quant = enable_fake_quant

    def forward(self, x):
        w = self.linear.weight
        b = self.linear.bias

        if self.enable_fake_quant:
            x_q = fake_quantize_tensor(x, self.num_bits)
            w_q = fake_quantize_tensor(w, self.num_bits)
            return F.linear(x_q, w_q, b)
        else:
            return self.linear(x)

    @property
    def weight(self):
        return self.linear.weight

    @property
    def bias(self):
        return self.linear.bias
def make_linear(config: Config):
    if config.use_fake_quant:
        return lambda in_f, out_f, bias=True: FakeQuantLinear(
            in_f, out_f, bias=bias, num_bits=config.fake_quant_bits, enable_fake_quant=True
        )
    else:
        return lambda in_f, out_f, bias=True: nn.Linear(in_f, out_f, bias=bias)




In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        assert config.d_model % config.n_heads == 0
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads
        self.scale = self.head_dim ** -0.5

        Linear = make_linear(config)

        self.q_proj = Linear(config.d_model, config.d_model)
        self.k_proj = Linear(config.d_model, config.d_model)
        self.v_proj = Linear(config.d_model, config.d_model)
        self.out_proj = Linear(config.d_model, config.d_model)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # [B, H, T, D]
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * self.scale  # [B, H, T, T]

        mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        att = att.masked_fill(~mask, float("-inf"))

        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v  # [B, H, T, D]
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))
        return y

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        Linear = make_linear(config)

        self.net = nn.Sequential(
            Linear(config.d_model, config.d_ff),
            nn.GELU(),
            Linear(config.d_ff, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)
    
class TransformerBlock(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.ff = FeedForward(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [5]:
class TinyTransformerLM(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config

        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.block_size, config.d_model)
        self.drop = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.ln_f = nn.LayerNorm(config.d_model)

        # keep lm head as nn.Linear so dynamic quantization can catch it easily
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # weight tying
        self.lm_head.weight = self.token_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, FakeQuantLinear):
            nn.init.normal_(module.linear.weight, mean=0.0, std=0.02)
            if module.linear.bias is not None:
                nn.init.zeros_(module.linear.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.block_size, "Sequence too long"

        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)  # [1, T]

        tok = self.token_emb(idx)         # [B, T, C]
        pos = self.pos_emb(pos)           # [1, T, C]
        x = self.drop(tok + pos)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)          # [B, T, vocab]

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=100, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)

        return idx


In [ ]:
class TinyTransformerLM(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config

        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.block_size, config.d_model)
        self.drop = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.ln_f = nn.LayerNorm(config.d_model)

        #  lm head as nn.Linear so dynamic quantization can catch 
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # weight tying
        self.lm_head.weight = self.token_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, FakeQuantLinear):
            nn.init.normal_(module.linear.weight, mean=0.0, std=0.02)
            if module.linear.bias is not None:
                nn.init.zeros_(module.linear.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.block_size, "Sequence too long"

        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)  # [1, T]

        tok = self.token_emb(idx)         # [B, T, C]
        pos = self.pos_emb(pos)           # [1, T, C]
        x = self.drop(tok + pos)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)          # [B, T, vocab]

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=100, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)

        return idx


# ============================================================
# 7. Evaluation / training
# ============================================================

@torch.no_grad()
def evaluate(model, loader, device, max_batches=50):
    model.eval()
    losses = []
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break
        x = x.to(device)
        y = y.to(device)
        _, loss = model(x, y)
        losses.append(loss.item())
    return sum(losses) / max(len(losses), 1)


def train_one_epoch(model, loader, optimizer, device, step_offset=0, log_every=100):
    model.train()
    running = 0.0
    for step, (x, y) in enumerate(loader):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running += loss.item()

        if (step + 1) % log_every == 0:
            avg = running / log_every
            print(f"step {step_offset + step + 1:6d} | train loss {avg:.4f}")
            running = 0.0


# ============================================================
# 8. Quantized inference export
# ============================================================

def export_dynamic_quantized_model(fp32_model: nn.Module):
    """
    Dynamic quantization of nn.Linear layers.
    This is the most practical built-in PyTorch baseline for CPU/edge inference.
    """
    fp32_model = fp32_model.cpu().eval()

    quantized_model = torch.quantization.quantize_dynamic(
        fp32_model,
        {nn.Linear},
        dtype=torch.qint8
    )
    return quantized_model


# ============================================================
# 9. Main
# ============================================================

def main():
    cfg = Config()
    set_seed(cfg.seed)

    book_path = Path(cfg.book_path)
    if not book_path.exists():
        url = "https://www.gutenberg.org/files/1342/1342-0.txt"
        print(f"{book_path} not found. Downloading from {url} ...")
        urllib.request.urlretrieve(url, book_path)
    text = book_path.read_text(encoding="utf-8")
    tokenizer = CharTokenizer(text)
    cfg.vocab_size = tokenizer.vocab_size
    cfg.d_ff = 4 * cfg.d_model

    print(f"Device: {cfg.device}")
    print(f"Characters in book: {len(text):,}")
    print(f"Vocab size: {cfg.vocab_size}")

    ids = tokenizer.encode(text)

    split_idx = int(len(ids) * cfg.train_split)
    train_ids = ids[:split_idx]
    val_ids = ids[split_idx:]

    train_ds = BookDataset(train_ids, cfg.block_size)
    val_ds = BookDataset(val_ids, cfg.block_size)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=True)

    model = TinyTransformerLM(cfg).to(cfg.device)

    print(f"Trainable params: {count_parameters(model):,}")
    print(f"Approx FP32 model size: {estimate_model_size_mb(model):.2f} MB")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )

    global_step = 0
    best_val = float("inf")

    for epoch in range(cfg.max_epochs):
        print(f"\n=== Epoch {epoch + 1}/{cfg.max_epochs} ===")
        start = time.time()

        train_one_epoch(model, train_loader, optimizer, cfg.device, step_offset=global_step, log_every=100)
        global_step += len(train_loader)

        train_loss = evaluate(model, train_loader, cfg.device, max_batches=50)
        val_loss = evaluate(model, val_loader, cfg.device, max_batches=50)

        elapsed = time.time() - start
        print(f"epoch {epoch + 1} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | time {elapsed:.1f}s")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": cfg.__dict__,
                    "stoi": tokenizer.stoi,
                    "itos": tokenizer.itos,
                },
                "best_fp32_model.pt"
            )
            print("Saved best FP32 checkpoint to best_fp32_model.pt")

    # --------------------------------------------------------
    # Text generation with FP32 / fake-quant-trained model
    # --------------------------------------------------------
    model.eval()
    prompt = "The "
    prompt_ids = tokenizer.encode(prompt)
    x = torch.tensor([prompt_ids], dtype=torch.long, device=cfg.device)

    out = model.generate(
        x,
        max_new_tokens=cfg.max_new_tokens,
        temperature=cfg.temperature,
        top_k=cfg.top_k
    )
    generated = tokenizer.decode(out[0].tolist())
    print("\n===== FP32/QAT MODEL GENERATION =====")
    print(generated)

    # --------------------------------------------------------
    # Export quantized inference model
    # --------------------------------------------------------
    quantized_model = export_dynamic_quantized_model(model)
    torch.save(
        {
            "model_state_dict": quantized_model.state_dict(),
            "config": cfg.__dict__,
            "stoi": tokenizer.stoi,
            "itos": tokenizer.itos,
        },
        "best_int8_dynamic_model.pt"
    )
    print("\nSaved quantized checkpoint to best_int8_dynamic_model.pt")

    print(f"Approx quantized model size (state_dict): {estimate_model_size_mb(quantized_model):.2f} MB")

    # --------------------------------------------------------
    # Quick CPU generation test with quantized model
    # --------------------------------------------------------
    quantized_model.eval()
    x_cpu = torch.tensor([prompt_ids], dtype=torch.long, device="cpu")
    with torch.no_grad():
        out_q = quantized_model.generate(
            x_cpu,
            max_new_tokens=cfg.max_new_tokens,
            temperature=cfg.temperature,
            top_k=cfg.top_k
        )
    generated_q = tokenizer.decode(out_q[0].tolist())
    print("\n===== INT8 DYNAMIC QUANTIZED MODEL GENERATION =====")
    print(generated_q)


if __name__ == "__main__":
    main()